In [ ]:
from typing import TypedDict, Optional, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

class ContactState(TypedDict):
    chat_history: str
    result: Dict[str, Any]
    action: Optional[str]  # 'confirm', 'edit', 'cancel'

# 2. นิยาม Nodes ต่างๆ
def extract_contact_node(state: ContactState) -> Dict[str, Any]:
    # เรียกใช้ฟังก์ชันเดิมของคุณ
    # result = summary_group_line_chat(state["chat_history"])
    result = {'companyTh': 'เอบีซี คอมพานี',
 'companyEn': 'ABC Company',
 'nameTh': 'พชร อุ้ยกิ้ม',
 'nameEn': 'Pachara Auikim',
 'nickname': 'บาส',
 'jobTitle': None,
 'phone': None,
 'email': None,
 'found': True}
    print("extract_contact_node\n")
    print(f"chat_history: {state['chat_history']}\naction:{state.get('action', "No action")}")
    return {"result": result}

def human_review_node(state: ContactState) -> Dict[str, Any]:
    print("\nhuman_review_node\n")
    result = state["result"]
    summary_text = "\n".join([
        f"{value_mapping.get(k, k)}: {result.get(k) or '<ไม่มีข้อมูล>'}"
        for k in value_mapping
    ])
    
    # 🛑 หยุดระบบชั่วคราว (PAUSE) ส่งข้อมูลไปให้ UI หรือ User ดู
    human_input = interrupt({
        "message": "โปรดตรวจสอบข้อมูลก่อนบันทึก",
        "summary": summary_text,
        "current_data": result
    })
    
    print(f"\n\nTEST: {human_input}\n\n")
    return {"action": human_input.get("action", "confirm")}

def save_to_neo4j_node(state: ContactState) -> Dict[str, Any]:

    result = state["result"]
    print(state.get("action", "no actoin"))
    # with db.get_session() as session:
    #     session.run(add_contact_query, **result)
    print("✅ บันทึกข้อมูลลง Neo4j เรียบร้อยแล้ว!")
    return {}

workflow = StateGraph(ContactState)

workflow.add_node("extract_contact", extract_contact_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("save_to_neo4j", save_to_neo4j_node)

workflow.add_edge(START, "extract_contact")
workflow.add_edge("extract_contact", "human_review")

def route_after_review(state: ContactState):
    if state.get("action") == "confirm" :
        return "save_to_neo4j"
    return END  

workflow.add_conditional_edges("human_review", route_after_review)
workflow.add_edge("save_to_neo4j", END)

checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
# 1. รันรอบแรกจนติด Interrupt
config = {"configurable": {"thread_id": "user_session_123"}}
stream = app.stream_events({"input": "data"}, config=config, version="v3")
# _ = stream.output  # รันลุยไฟไปจนติด Interrupt หรือจบ

if stream.interrupted:
    print(stream.interrupts)  # ดึงข้อมูล interrupt ออกมาดูได้เลย
    
    user_choice = input("Input: ")
            
    if user_choice == '1':
                response_payload = {"action": "confirm"}
    else:
                response_payload = {"action": "cancel"}
            
            # เตรียม Command เพื่อส่งให้ app.stream ในรอบถัดไปของ while loop
    # current_input = Command(resume=response_payload)
    
    resumed = app.stream_events(Command(resume=response_payload), config=config, version="v3")
    final_state = resumed.output  # รันต่อจนจบแล้วดึง State ล่าสุดออกมา